In [0]:
-- ─── Avec Unity Catalog : CREATE SCHEMA (3 niveaux : catalogue.schéma.table) ──
-- Le catalogue par défaut est hive_metastore.
-- On crée un schéma par couche médaillon, chacun pointant vers son External Location.

CREATE SCHEMA IF NOT EXISTS hive_metastore.bronze
  COMMENT 'Couche Bronze : données brutes'
  MANAGED LOCATION 'abfss://bronze@adlscommercialprod.dfs.core.windows.net/';

CREATE SCHEMA IF NOT EXISTS hive_metastore.silver
  COMMENT 'Couche Silver : données nettoyées'
  MANAGED LOCATION 'abfss://silver@adlscommercialprod.dfs.core.windows.net/';

CREATE SCHEMA IF NOT EXISTS hive_metastore.gold
  COMMENT 'Couche Gold : données curated'
  MANAGED LOCATION 'abfss://gold@adlscommercialprod.dfs.core.windows.net/';

-- Vérification
SHOW SCHEMAS IN hive_metastore;

databaseName
bronze
default
gold
silver


In [0]:
%python
       
from pyspark.sql.functions import current_timestamp, lit, col
from datetime import datetime

# Unity Catalog : notation 3 niveaux catalogue.schéma.table
CATALOG      = "hive_metastore"
SOURCE_PATH  = "abfss://landing@adlscommercialprod.dfs.core.windows.net/ventes commerciales.csv"
BRONZE_TABLE = f"{CATALOG}.bronze.ventes_raw"
BRONZE_PATH  = "abfss://bronze@adlscommercialprod.dfs.core.windows.net/ventes_raw"

# 1. Lecture CSV — tout en STRING (règle Bronze)
df_raw = (
    spark.read
    .option("header",      "true")
    .option("inferSchema", "false")
    .option("delimiter",   ",")
    .option("encoding",    "UTF-8")
    .csv(SOURCE_PATH)
)

# 2. Ajout métadonnées d'audit
df_bronze = (
    df_raw
    .withColumn("_ingestion_ts", current_timestamp())
    .withColumn("_source_file",  col("_metadata.file_path"))
    .withColumn("_batch_id",     lit(datetime.now().strftime("%Y%m%d_%H%M%S")))
)

# 3. Écriture Delta en mode append
(
    df_bronze.write
    .format("delta")
    .mode("append")
    .option("path", BRONZE_PATH)
    .option('mergeSchema', 'true').saveAsTable(BRONZE_TABLE)
)

print(f"✅ Bronze chargé : {df_bronze.count()} lignes")
df_bronze.printSchema()

✅ Bronze chargé : 8 lignes
root
 |-- order_id: string (nullable = true)
 |-- date_commande: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- nom_client: string (nullable = true)
 |-- produit_id: string (nullable = true)
 |-- produit: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- region: string (nullable = true)
 |-- qte: string (nullable = true)
 |-- prix_unitaire: string (nullable = true)
 |-- remise: string (nullable = true)
 |-- _ingestion_ts: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- _batch_id: string (nullable = false)



In [0]:
SELECT * FROM hive_metastore.bronze.ventes_raw LIMIT 5;

order_id,date_commande,client_id,nom_client,produit_id,produit,categorie,region,qte,prix_unitaire,remise,_ingestion_ts,_source_file,_batch_id
1,2024-01-15,C001,Dupont SA,P001,Laptop Pro,INFORMATIQUE,Ile-de-France,2,1200.00,0.10,2026-04-18T14:07:15.654Z,null,20260418_140711
2,2024-01-22,C002,Martin SARL,P002,Clavier mecanique,PERIPHERIQUES,Auvergne-Rhone-Alpes,5,89.99,0.00,2026-04-18T14:07:15.654Z,null,20260418_140711
3,2024-02-03,C001,Dupont SA,P003,Moniteur 4K,INFORMATIQUE,Ile-de-France,1,699.00,0.05,2026-04-18T14:07:15.654Z,null,20260418_140711
4,2024-02-17,C003,Leblanc Fils,P001,Laptop Pro,INFORMATIQUE,Bretagne,3,1200.00,0.15,2026-04-18T14:07:15.654Z,null,20260418_140711
5,2024-03-01,C002,Martin SARL,P004,Webcam HD,PERIPHERIQUES,Auvergne-Rhone-Alpes,2,149.99,0.00,2026-04-18T14:07:15.654Z,null,20260418_140711
